# 🎙️ IndexTTS2 Lightweight English Model Training on Kaggle

Welcome! This notebook is configured to run on Kaggle's free GPUs (P100 or T4 x2). 
Make sure you head to the sidebar (Settings / Accelerator) and turn on a GPU before starting.

### Step 1: Clone Your Repository
We will pull your exact code and lightweight setup directly from GitHub.

In [ ]:
!git clone https://github.com/dipankarhandique340/IndexTTS3.git
%cd IndexTTS3

### Step 2: Set Up Environment & Requirements
We will install the required PyTorch packages, DeepSpeed for optimized training, and other audio dependencies.

In [ ]:
# Install Flash Attention & dependencies
!pip install torch==2.5.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install flash-attn --no-build-isolation
!pip install deepspeed
!pip install uv
!uv pip install --system -r requirements.txt

### Step 3: Generate the Fresh Lightweight Checkpoints
Since the large `.pth` checkpoint files are ignored by GitHub, we need to quickly re-generate your empty, lightweight Neural Network architecture right here in Kaggle so we have a blank slate to train on.

In [ ]:
!cd training_lightweight_en && python init_model.py
!ls -lh training_lightweight_en/*.pth

### Step 4: Download & Extract Your Dataset
Instead of uploading files manually through Kaggle, we can pull them directly from a direct download link (e.g., a HuggingFace dataset link, a direct Google Drive link, or your own server).
1. Paste your direct download link below.
2. The script will download the zip file to `datasets/` and extract your `.jsonl` and `.wav` audio files automatically.

In [ ]:
# Replace this URL with the direct link to your dataset zip file
DATASET_URL="https://huggingface.co/datasets/YOUR_USERNAME/YOUR_DATASET/resolve/main/dataset.zip"

# Create directory and download the dataset
!mkdir -p datasets
!wget -O datasets/my_dataset.zip $DATASET_URL

# Extract the audio and JSONL files
!unzip -q datasets/my_dataset.zip -d datasets/
!echo "✅ Dataset downloaded and extracted successfully!"

### Step 5: Train the Lightweight GPT (Text-to-Semantic)
This begins the training for the ~1.7 GB GPT component. Modify the `--train-manifest` and `--val-manifest` lines below before running!

In [ ]:
%%writefile training_lightweight_en/train_lightweight_gpt.sh
#!/bin/bash
echo "Starting GPT lightweight training..."
python trainers/train_gpt_v2.py \
    --train-manifest datasets/english_train_data.jsonl \
    --val-manifest datasets/english_val_data.jsonl \
    --tokenizer checkpoints/bpe.model \
    --config training_lightweight_en/config_light.yaml \
    --base-checkpoint training_lightweight_en/gpt_light_init.pth \
    --output-dir checkpoints_lightweight_gpt \
    --batch-size 32 \
    --grad-accumulation 1 \
    --epochs 50 \
    --learning-rate 1e-4 \
    --weight-decay 0.01 \
    --warmup-steps 1000 \
    --log-interval 1 \
    --val-interval 2000 \
    --grad-clip 1.0 \
    --text-loss-weight 0.2 \
    --mel-loss-weight 0.8 \
    --amp

In [ ]:
!chmod +x training_lightweight_en/train_lightweight_gpt.sh
!bash training_lightweight_en/train_lightweight_gpt.sh

### Step 6: Train the Lightweight DiT / S2Mel Component
Once the GPT model finishes training, you'll need to train S2Mel to accurately convert those semantic tokens into raw Mel Spectrograms.

In [ ]:
%%writefile training_lightweight_en/train_lightweight_s2mel.sh
#!/bin/bash
echo "Starting S2Mel lightweight training..."
python trainers/train_s2mel_v2.py \
    --train-manifest datasets/english_train_data.jsonl \
    --val-manifest datasets/english_val_data.jsonl \
    --config training_lightweight_en/config_light.yaml \
    --base-checkpoint training_lightweight_en/s2mel_light_init.pth \
    --output-dir checkpoints_lightweight_s2mel \
    --batch-size 16 \
    --grad-accumulation 2 \
    --epochs 100 \
    --learning-rate 1e-4 \
    --val-interval 2000 \
    --amp

In [ ]:
!chmod +x training_lightweight_en/train_lightweight_s2mel.sh
!bash training_lightweight_en/train_lightweight_s2mel.sh

### Congratulations! 🎉
You have successfully trained your lightweight voice clones. Save the output models located in `checkpoints_lightweight_gpt` and `checkpoints_lightweight_s2mel` (found inside the `/kaggle/working/IndexTTS3/` output panel on the right) back to your local computer.